# Exploracao do dataset PlantVillage

Este notebook cobre a etapa de **compreensao e auditoria dos dados** do TCC.

Nesta etapa vamos apenas:

- montar o Google Drive, quando estiver no Colab;
- preparar as pastas do projeto;
- baixar os arquivos oficiais `data.zip` e `leaf_grouping/leaf-map.json`;
- ler somente as imagens em `raw/color/` diretamente do ZIP;
- associar cada imagem ao `leaf_id`;
- gerar e salvar um CSV de metadados.

**Ainda nao sera feita divisao treino/validacao/teste, data augmentation nem treinamento de modelos.**

In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

In [ ]:
if IN_COLAB:
    BASE_DIR = Path("/content/drive/MyDrive/TCC")
else:
    current_dir = Path.cwd().resolve()
    BASE_DIR = current_dir.parent if current_dir.name == "notebooks" else current_dir

DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"
SRC_DIR = BASE_DIR / "src"

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Pasta base:", BASE_DIR)
print("Pasta dos dados:", DATA_DIR)
print("Pasta de resultados:", RESULTS_DIR)
print("Pasta src:", SRC_DIR)

## Download dos arquivos oficiais

O repositorio do PlantVillage no Hugging Face disponibiliza o arquivo `data.zip` com as imagens e o arquivo `leaf_grouping/leaf-map.json` com o mapa usado para agrupar imagens derivadas da mesma folha.

O agrupamento por folha sera importante depois para evitar vazamento de dados entre treino, validacao e teste.

In [ ]:
try:
    from huggingface_hub import hf_hub_download
except ImportError:
    %pip install -q huggingface_hub
    from huggingface_hub import hf_hub_download

In [ ]:
zip_path = Path(
    hf_hub_download(
        repo_id="mohanty/PlantVillage",
        filename="data.zip",
        repo_type="dataset",
        local_dir=str(DATA_DIR),
    )
)

leaf_map_path = Path(
    hf_hub_download(
        repo_id="mohanty/PlantVillage",
        filename="leaf_grouping/leaf-map.json",
        repo_type="dataset",
        local_dir=str(DATA_DIR),
    )
)

print("Arquivo de imagens:", zip_path)
print("Mapa de folhas:", leaf_map_path)

## Verificacao do download

In [ ]:
import os
import zipfile

zip_size_gb = os.path.getsize(zip_path) / (1024 ** 3)

with zipfile.ZipFile(zip_path, "r") as zip_file:
    total_zip_members = len(zip_file.infolist())

print(f"Tamanho do data.zip: {zip_size_gb:.2f} GB")
print("data.zip existe:", zip_path.exists())
print("leaf-map.json existe:", leaf_map_path.exists())
print("Quantidade total de entradas no ZIP:", total_zip_members)

## Metadados de `raw/color/`

As funcoes abaixo leem o indice do ZIP e o `leaf-map.json`, sem extrair as imagens. Cada linha representa uma imagem em `raw/color/`.

In [ ]:
from plantvillage_audit import (
    audit_metadata,
    build_metadata_dataframe,
    save_metadata_csv,
)

metadata = build_metadata_dataframe(
    zip_path=zip_path,
    leaf_map_path=leaf_map_path,
)

print("Imagens encontradas em raw/color/:", len(metadata))
metadata.head()

## Auditoria do dataset

In [ ]:
audit = audit_metadata(metadata)

display(audit["resumo"])

In [ ]:
display(audit["imagens_por_classe"])

In [ ]:
display(audit["status_leaf_id"])

## Imagens sem `leaf_id`

Falhas de associacao permanecem explicitas no dataframe por meio das colunas `leaf_id_found` e `leaf_match_status`.

In [ ]:
falhas_leaf_id = metadata.loc[
    ~metadata["leaf_id_found"],
    ["zip_path", "classe", "filename", "leaf_lookup_key", "leaf_match_status"],
]

print("Imagens sem leaf_id encontrado:", len(falhas_leaf_id))
falhas_leaf_id.head(20)

## CSV de metadados

O CSV e salvo em `results/`, pasta ignorada pelo Git neste projeto.

In [ ]:
metadata_csv_path = save_metadata_csv(
    metadata=metadata,
    output_path=RESULTS_DIR / "plantvillage_metadata_raw_color.csv",
)

print("CSV salvo em:", metadata_csv_path)